<a href="https://colab.research.google.com/github/ch-jsk/jsk-flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ch-jsk/jsk-flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research question

Can a lightweight machine-learning ranking help prioritize webpages for human review when their observed click-through rate (CTR) falls below the typical CTR of pages in a similar search-position tier?

### Decision to support

The intended decision is which eligible pages should be reviewed first—not which pages should automatically be changed. The ranking is designed to help SEO or content teams allocate limited review time.

### Hypothesis

A Random Forest using prior-period search performance and content metadata may rank below-reference pages more effectively than a simple baseline based on prior CTR shortfall.

### Scope

This is an observational decision-support experiment. It evaluates ranking performance on held-out clients and a later outcome month. It does not estimate the causal impact of content edits, guarantee future CTR improvements, or establish that a page has a content-quality problem.

In [ ]:
# Section 1 — Research question and experiment scope

research_question = (
    "Can a lightweight ML ranking prioritize pages whose CTR falls "
    "below a position-tier reference for human review?"
)

decision_supported = "Prioritize pages for human review"
prediction_target = (
    "Outcome-period CTR below the frozen training-client "
    "CTR median for the corresponding position tier"
)

print("Research question:", research_question)
print("Decision supported:", decision_supported)
print("Prediction target:", prediction_target)
print("Claim boundary: observational ranking; no causal claim.")

Research question: Can a lightweight ML ranking prioritize pages whose CTR falls below a position-tier reference for human review?
Decision supported: Prioritize pages for human review
Prediction target: Outcome-period CTR below the frozen training-client CTR median for the corresponding position tier
Claim boundary: observational ranking; no causal claim.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data source and grain

The experiment uses the FlyRank internship warehouse, accessed through DuckDB over remote Parquet files. The daily performance fact table is aggregated to a client–content pair for each observation month. Pseudonymous identifiers are used only to join and group records; they are not model features or published recommendations.

### Development and evaluation periods

- **March 2026:** development features and initial ranking baseline.
- **April 2026:** development outcome used to define the target.
- **May 2026:** frozen-model feature snapshot.
- **June 2026:** sealed final outcome month, accessed only after the development procedure was fixed.

The final evaluation uses the eight held-out clients from the client-level split. Pages must have measurable June CTR to receive a June outcome label.

### Data quality and availability

Metrics are aggregated only where the corresponding data-availability flag is true. CTR is calculated as clicks divided by impressions; it is undefined when impressions are zero. Weighted search position uses impression-weighted position information where available.

The May feature cohort contained 237,910 page pairs with measurable CTR and position across 56 clients. The final May cohort contained 23,402 rows across the eight held-out clients before requiring a measurable June outcome. The final evaluation included 21,548 pairs.

### Privacy and data-use boundaries

No client names, domains, URLs, search queries, raw identifiers, or page-level recommendation lists are included in the public report. Results are presented as aggregate metrics.

In [ ]:
# Section 2 — Data contract and aggregate cohort checks

data_contract = {
    "source": "FlyRank internship warehouse",
    "access_method": "DuckDB over remote Parquet",
    "development_features": "March 2026",
    "development_outcomes": "April 2026",
    "final_features": "May 2026",
    "sealed_final_outcomes": "June 2026",
    "split_unit": "client_hash_id",
    "zero_impression_ctr": "undefined, not zero",
    "public_output_policy": "aggregate metrics only",
}

print("Data contract")
for key, value in data_contract.items():
    print(f"- {key}: {value}")

print("\nObserved aggregate counts")
print("- March development pairs:", 331437)
print("- March-April eligible pairs before split:", 74502)
print("- Development training pairs:", 61874)
print("- Development held-out pairs:", 11338)
print("- May feature pairs with measurable CTR:", 237910)
print("- Final May held-out cohort:", 23402)
print("- Final pairs with measurable June CTR:", 21548)

Data contract
- source: FlyRank internship warehouse
- access_method: DuckDB over remote Parquet
- development_features: March 2026
- development_outcomes: April 2026
- final_features: May 2026
- sealed_final_outcomes: June 2026
- split_unit: client_hash_id
- zero_impression_ctr: undefined, not zero
- public_output_policy: aggregate metrics only

Observed aggregate counts
- March development pairs: 331437
- March-April eligible pairs before split: 74502
- Development training pairs: 61874
- Development held-out pairs: 11338
- May feature pairs with measurable CTR: 237910
- Final May held-out cohort: 23402
- Final pairs with measurable June CTR: 21548


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Approach

I used a Random Forest classifier to rank pages that may underperform relative to a position-tier CTR reference. The model was chosen because it can capture nonlinear relationships among prior search performance and content metadata without requiring a linear relationship between features and the target.

### Target definition

For development, a page is labelled positive when its April 2026 CTR is below the April CTR median calculated from training clients in the same March position tier. This creates a relative underperformance label; it does not establish that a page is objectively poor or that editing it will improve CTR.

### Features

The model uses nine March-derived features:

- Log-transformed March impressions
- March CTR
- March impression-weighted average search position
- Number of days with GSC data available
- Search-volume metadata
- Word count
- March position tier
- Content type
- Main intent

Numeric missing values are imputed using the training-set median. Missing categorical values are imputed using the training-set most frequent category, followed by one-hot encoding. Unseen categories are handled without failing the prediction pipeline.

Pseudonymous client and content IDs are used for joining, grouping, and splitting only. They are not model features. April outcomes are used to construct labels, not as model inputs.

### Baseline

The baseline ranks pages using the positive shortfall between March CTR and the training clients' March median CTR for the same March position tier. Both methods are evaluated on the same held-out client set using Precision@10%, recall, lift over the positive-label base rate, and the mean observed CTR gap among shortlisted pages.

### Validation design

The development split is grouped by client: 29 training clients and 8 held-out clients. This tests generalization to clients not used to fit the model. After development choices were recorded, the frozen model was scored using May features and evaluated against June outcomes for the held-out clients. The June outcome month was not used to train or tune the model.

The June evaluation includes only pages with measurable June CTR. Results therefore apply to this evaluated cohort, not automatically to all pages in the warehouse.

In [ ]:
# Section 3 — Methodology configuration and split checks

method_config = {
    "model": "RandomForestClassifier",
    "n_estimators": 200,
    "min_samples_leaf": 10,
    "class_weight": "balanced",
    "random_state": 42,
    "split_unit": "client_hash_id",
    "training_clients": 29,
    "held_out_clients": 8,
    "training_page_pairs": 61874,
    "development_test_page_pairs": 11338,
    "numeric_features": [
        "log_impressions_march",
        "ctr_march_pct",
        "weighted_position_march",
        "gsc_days_available",
        "search_volume",
        "word_count",
    ],
    "categorical_features": [
        "march_position_tier",
        "content_type",
        "main_intent",
    ],
}

assert (
    method_config["training_page_pairs"]
    + method_config["development_test_page_pairs"]
    == 73212
)

print("Model:", method_config["model"])
print("Random seed:", method_config["random_state"])
print("Training clients:", method_config["training_clients"])
print("Held-out clients:", method_config["held_out_clients"])
print("Training page pairs:", method_config["training_page_pairs"])
print("Development test page pairs:",
      method_config["development_test_page_pairs"])
print("Total model-cohort page pairs:",
      method_config["training_page_pairs"]
      + method_config["development_test_page_pairs"])
print("Feature count:",
      len(method_config["numeric_features"])
      + len(method_config["categorical_features"]))

Model: RandomForestClassifier
Random seed: 42
Training clients: 29
Held-out clients: 8
Training page pairs: 61874
Development test page pairs: 11338
Total model-cohort page pairs: 73212
Feature count: 9


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Development evaluation: March features → April outcomes

The development evaluation used 11,338 page pairs from eight held-out clients. The positive-label base rate was 51.52%.

| Metric | March CTR baseline | Random Forest |
|---|---:|---:|
| Precision@10% | 80.69% | **85.19%** |
| Lift over base rate | 1.566× | **1.654×** |
| Recall@10% | 15.67% | **16.54%** |
| Mean observed CTR gap in top 10%, percentage points | 0.1195 | 0.0887 |

The Random Forest improved Precision@10% by 4.50 percentage points over the baseline. Its ROC-AUC was 0.8158 and average precision was 0.7903.

### Final temporal evaluation: May features → June outcomes

The frozen model was evaluated on 21,548 page pairs from the same eight held-out clients for which June CTR was measurable. The positive-label base rate was 55.35%.

| Metric | May CTR baseline | Frozen Random Forest |
|---|---:|---:|
| Precision@10% | 80.65% | **82.55%** |
| Lift over base rate | 1.457× | **1.491×** |
| Recall@10% | 14.57% | **14.92%** |
| Mean observed CTR gap in top 10%, percentage points | 0.1236 | 0.0582 |

The model's Precision@10% advantage decreased from 4.50 percentage points during development to 1.90 percentage points in the final evaluation. Its June ROC-AUC was 0.7908 and average precision was 0.7784.

### Position-tier results

The June tier-level comparison was mixed. The model had slightly higher Precision@10% for positions 11–20 (76.53% versus 76.24%) and positions 21–50 (86.88% versus 86.05%). The baseline performed better for positions 4–10 (81.53% versus 74.29%) and the top 3 (100.00% versus 80.77%).

The top-3 result is based on only 255 evaluated page pairs and a shortlist of 26, so it should be interpreted cautiously.

### Interpretation

The frozen model retained a modest overall Precision@10% advantage over the baseline on the June cohort, but the advantage varied by position tier. The baseline selected pages with a larger mean observed CTR shortfall, while the model's shortlist had slightly higher precision. These are different review priorities, and neither result establishes that changing a page will cause its CTR to improve.

In [ ]:
# Section 4 — Aggregate results and consistency checks

development_results = {
    "base_rate_pct": 51.52,
    "baseline_precision_at_10_pct": 80.69,
    "model_precision_at_10_pct": 85.19,
    "baseline_lift": 1.566,
    "model_lift": 1.654,
    "baseline_recall_at_10_pct": 15.67,
    "model_recall_at_10_pct": 16.54,
    "model_roc_auc": 0.8158,
    "model_average_precision": 0.7903,
}

final_results = {
    "base_rate_pct": 55.35,
    "baseline_precision_at_10_pct": 80.65,
    "model_precision_at_10_pct": 82.55,
    "baseline_lift": 1.457,
    "model_lift": 1.491,
    "baseline_recall_at_10_pct": 14.57,
    "model_recall_at_10_pct": 14.92,
    "model_roc_auc": 0.7908,
    "model_average_precision": 0.7784,
}

development_precision_gain = (
    development_results["model_precision_at_10_pct"]
    - development_results["baseline_precision_at_10_pct"]
)

final_precision_gain = (
    final_results["model_precision_at_10_pct"]
    - final_results["baseline_precision_at_10_pct"]
)

print("Development positive-label base rate:",
      development_results["base_rate_pct"], "%")
print("Development Precision@10% gain:",
      round(development_precision_gain, 2), "percentage points")
print("Final positive-label base rate:",
      final_results["base_rate_pct"], "%")
print("Final Precision@10% gain:",
      round(final_precision_gain, 2), "percentage points")
print("Final ROC-AUC:", final_results["model_roc_auc"])
print("Final average precision:",
      final_results["model_average_precision"])

assert round(development_precision_gain, 2) == 4.50
assert round(final_precision_gain, 2) == 1.90
assert final_results["model_roc_auc"] > 0.5

print("Aggregate result checks passed.")

Development positive-label base rate: 51.52 %
Development Precision@10% gain: 4.5 percentage points
Final positive-label base rate: 55.35 %
Final Precision@10% gain: 1.9 percentage points
Final ROC-AUC: 0.7908
Final average precision: 0.7784
Aggregate result checks passed.


## 5. Limitations

*What this work cannot claim.*

### Limitations and threats to validity

1. **Observational evidence:** The analysis identifies pages associated with below-reference click-through rates (CTR). It does not establish that any particular page characteristic caused the underperformance.

2. **Reference-based labels:** A page is labelled as underperforming when its outcome-month CTR falls below the frozen reference median for its position tier. This is a relative benchmark, not a universal definition of poor performance.

3. **Position-tier differences:** The model's performance varies across position tiers. In the June evaluation, it performed better overall than the CTR-gap baseline, but the baseline performed better in the positions 4–10 and top-3 tiers. The top-3 evaluation also has a small sample size.

4. **Temporal and distribution shift:** The model was trained using March features, evaluated during development on April outcomes, and subsequently evaluated on May features against June outcomes without retraining. Changes in search demand, page composition, or ranking distributions may affect performance.

5. **Incomplete measurement:** Some page-month observations lack measurable impressions or CTR and cannot be evaluated. Missing data must not be interpreted as zero performance.

6. **Client-level generalization:** Evaluation uses held-out clients, but the results may not generalize to every client, industry, website, or future period.

7. **Operational validation:** The ranking is a decision-support signal, not an automatic instruction to change content. A prospective human-reviewed trial is needed to determine whether using the ranking improves outcomes.


In [ ]:
# Section 5: Limitations — reported evaluation checks

june_evaluated_pages = 21_548
june_held_out_clients = 8

june_baseline_precision_at_10 = 0.8065
june_model_precision_at_10 = 0.8255

assert june_evaluated_pages > 0
assert june_held_out_clients > 0
assert june_model_precision_at_10 > june_baseline_precision_at_10

print("Reported June evaluation checks passed.")
print(f"Evaluated page pairs: {june_evaluated_pages:,}")
print(f"Held-out clients: {june_held_out_clients}")
print(
    "Precision@10%: "
    f"baseline={june_baseline_precision_at_10:.2%}, "
    f"model={june_model_precision_at_10:.2%}"
)
print("These checks validate recorded results, not a fresh experiment run.")

Reported June evaluation checks passed.
Evaluated page pairs: 21,548
Held-out clients: 8
Precision@10%: baseline=80.65%, model=82.55%
These checks validate recorded results, not a fresh experiment run.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### Ranked recommendations

Based on the June held-out evaluation, the following actions are recommended in priority order.

1. **Pilot a human-reviewed priority queue.** Use the model to help identify pages for review, not to trigger automatic content changes. The model achieved 82.55% Precision@10%, compared with 80.65% for the CTR-gap baseline across the evaluated pages.

2. **Use position-tier-specific ranking policies.** The model performed better overall, but the baseline performed better in the positions 4–10 and top-3 tiers. Evaluate a hybrid approach that uses the baseline for these tiers and the model for positions 11–50. This is a candidate policy to test, not a proven improvement.

3. **Review pages before recommending changes.** Human reviewers should investigate content relevance, search intent, snippet quality, competition, and measurement coverage before deciding what action—if any—is appropriate.

4. **Run a prospective evaluation.** Freeze the ranking rules before the next evaluation period. Compare the model, baseline, and proposed hybrid policy on a later outcome period without tuning against that period.

5. **Monitor performance and data quality.** Track Precision@10%, recall, lift, measurable-page coverage, and performance by position tier. Reassess the approach if results deteriorate or the data distribution changes.

**Decision:** The model is a promising supplementary prioritization signal, but the evidence does not yet justify replacing the baseline across every position tier or claiming that model-guided edits will improve CTR.


In [ ]:
# Section 6: Ranked recommendation checks

baseline_precision_at_10 = 0.8065
model_precision_at_10 = 0.8255

# June tier results: baseline performed better in positions 4–10 and top 3.
baseline_precision_positions_4_10 = 0.8153
model_precision_positions_4_10 = 0.7429

baseline_precision_top_3 = 1.0000
model_precision_top_3 = 0.8077

assert model_precision_at_10 > baseline_precision_at_10
assert baseline_precision_positions_4_10 > model_precision_positions_4_10
assert baseline_precision_top_3 > model_precision_top_3

print("Recommendation checks passed.")
print("Use the model as supplementary decision support.")
print("Test the proposed hybrid policy prospectively before adopting it.")

Recommendation checks passed.
Use the model as supplementary decision support.
Test the proposed hybrid policy prospectively before adopting it.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

### Artifacts embedded in the research paper

The paper should include or reference the following artifacts so readers can inspect the evaluation and understand how the conclusions were reached.

1. **Evaluation summary:** Development and June held-out performance for the CTR-gap baseline and Random Forest model, including Precision@10%, lift, recall, and ranking quality.
2. **Position-tier comparison:** June results for positions 11–20, 21–50, 4–10, and top 3, highlighting where the model does and does not outperform the baseline.
3. **Methodology and data protocol:** Cohort eligibility, client-level train/test separation, feature definitions, missing-data handling, frozen reference medians, and the sealed June evaluation protocol.
4. **Reproducibility record:** Aggregate experiment configuration and results, without client identifiers, page URLs, private search queries, or raw client data.
5. **Limitations and decision guidance:** The observational nature of the findings, variation across position tiers, and the need for prospective human-reviewed validation.

**Data safety:** Only aggregate metrics and non-sensitive methodological details should be published. Do not commit raw warehouse extracts, client names, domains, page URLs, private queries, or identifiable page/client-level predictions.


In [ ]:
# Section 7: Verify the aggregate development record

from pathlib import Path
import json

record_path = Path("/content/jsk-flyrank-ml-internship/work/notebooks/development_record.json")

assert record_path.exists(), f"Development record not found: {record_path}"

with record_path.open("r", encoding="utf-8") as f:
    development_record = json.load(f)

assert isinstance(development_record, dict), "Expected a JSON object."

print("Aggregate development record found and parsed.")
print(f"File: {record_path.name}")
print("Before committing, verify it contains aggregate metrics only.")

Aggregate development record found and parsed.
File: development_record.json
Before committing, verify it contains aggregate metrics only.


## ML-12: Demo and communication

### Five-minute demo outline

**0:00–0:45 — Problem and decision**
Explain why pages that underperform relative to comparable search-position tiers deserve human review, and clarify that the project ranks review opportunities rather than predicting the causal effect of edits.

**0:45–1:30 — Data and methodology**
Describe the warehouse data, March feature snapshot, April development outcomes, client-level split, CTR-gap baseline, and Random Forest model.

**1:30–2:30 — Development results**
Compare the baseline and model using Precision@10%, lift, recall, and ranking quality.

**2:30–3:30 — June held-out evaluation**
Present the frozen evaluation results and explain how performance varies by position tier.

**3:30–4:15 — Recommendations and limitations**
Discuss the proposed human-reviewed queue, the potential hybrid policy, missing measurements, and the need for prospective validation.

**4:15–5:00 — Reproducibility and next steps**
Show the paper, aggregate experiment record, data-safety precautions, and the proposed next evaluation.

### Social post

I completed an ML capstone on ranking pages for human review when their click-through rates fall below a reference benchmark for comparable search-position tiers.

I compared a simple CTR-gap baseline with a Random Forest model and evaluated the frozen approach on a later held-out outcome period. The model improved overall Precision@10% in the June evaluation, although results varied by position tier.

The main takeaway: a stronger overall ranking is not automatically the best choice for every segment. The next step is to test a human-reviewed or hybrid prioritization policy prospectively.

Read the paper: https://ch-jsk.github.io/jsk-flyrank-ml-internship/

### Employer-facing summary

I built and evaluated a machine-learning ranking approach to prioritize pages for review based on below-reference click-through rates, using a CTR-gap baseline and a Random Forest model. I built and evaluated a Random Forest ranking approach to help prioritize SEO pages for human review when observed click-through rate falls below a position-tier reference. Using FlyRank internship warehouse data, I trained on March features with April development outcomes and evaluated a frozen model on **21,548 measurable page pairs from a May-to-June temporal evaluation across held-out client groups**. The model achieved **82.55% precision among the top 10% of ranked candidates versus 80.65% for the baseline**, but its advantage varied by position tier. I recommend treating the model as decision support and validating a tier-aware, human-reviewed workflow before operational adoption.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
